# Avaliacao das planilhas ANP

Este notebook consolida as planilhas semanais de revenda da ANP baixadas em `temp/combustiveis/planilhas_originais`, filtra os produtos de interesse e calcula a media de `PRECO DE REVENDA` por `data_planilha`, `estado`, `cidade` e `produto`.

O resultado final e salvo em `temp/combustiveis/saida/planilha_consolidada.csv`.

In [1]:
from pathlib import Path
import re

import pandas as pd

cwd = Path.cwd().resolve()
candidatos = [cwd / "temp" / "combustiveis"]

if cwd.name == "levantamento" and cwd.parent.name == "combustiveis":
    candidatos.append(cwd.parent)
if cwd.name == "combustiveis":
    candidatos.append(cwd)

base_dir = next(
    (candidato for candidato in candidatos if (candidato / "planilhas_originais").exists()),
    None,
)

if base_dir is None:
    raise FileNotFoundError(
        "Nao encontrei a pasta temp/combustiveis/planilhas_originais a partir do diretorio atual."
    )

input_dir = base_dir / "planilhas_originais"
output_dir = base_dir / "saida"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "planilha_consolidada.csv"

produtos_interesse = {"ETANOL", "GLP", "GASOLINA COMUM", "GASOLINA ADITIVADA"}

input_dir, output_path

(PosixPath('/home/mbmn/dev/ando/organizador/temp/combustiveis/planilhas_originais'),
 PosixPath('/home/mbmn/dev/ando/organizador/temp/combustiveis/saida/planilha_consolidada.csv'))

In [2]:
def extrair_data_planilha(nome_arquivo: str) -> pd.Timestamp:
    match = re.search(
        r"revendas_lpc_\d{4}-\d{2}-\d{2}_(\d{4}-\d{2}-\d{2})\.xlsx$",
        nome_arquivo,
    )
    if not match:
        raise ValueError(f"Nome de arquivo fora do padrao esperado: {nome_arquivo}")
    return pd.to_datetime(match.group(1))


def carregar_planilha(path: Path) -> pd.DataFrame:
    data_planilha = extrair_data_planilha(path.name)

    # Nas planilhas da ANP, a linha 10 contem o cabecalho tabular.
    df = pd.read_excel(path, header=9)
    df.columns = [str(col).strip() for col in df.columns]

    df = df[["ESTADO", "MUNICIPIO" if "MUNICIPIO" in df.columns else "MUNICÍPIO", "PRODUTO", "PREÇO DE REVENDA"]].copy()
    df.columns = ["estado", "cidade", "produto", "preco_revenda"]

    df["estado"] = df["estado"].astype(str).str.strip()
    df["cidade"] = df["cidade"].astype(str).str.strip()
    df["produto"] = df["produto"].astype(str).str.strip()
    df["preco_revenda"] = pd.to_numeric(df["preco_revenda"], errors="coerce")
    df["data_planilha"] = data_planilha

    df = df[df["produto"].isin(produtos_interesse)]
    df = df.dropna(subset=["estado", "cidade", "produto", "preco_revenda"])

    return df[["data_planilha", "estado", "cidade", "produto", "preco_revenda"]]


arquivos = sorted(input_dir.glob("revendas_lpc_*.xlsx"))
len(arquivos), arquivos[:3]

(11,
 [PosixPath('/home/mbmn/dev/ando/organizador/temp/combustiveis/planilhas_originais/revendas_lpc_2025-12-28_2026-01-03.xlsx'),
  PosixPath('/home/mbmn/dev/ando/organizador/temp/combustiveis/planilhas_originais/revendas_lpc_2026-01-04_2026-01-10.xlsx'),
  PosixPath('/home/mbmn/dev/ando/organizador/temp/combustiveis/planilhas_originais/revendas_lpc_2026-01-11_2026-01-17.xlsx')])

In [3]:
dados_brutos = pd.concat([carregar_planilha(path) for path in arquivos], ignore_index=True)

resultado = (
    dados_brutos.groupby(["data_planilha", "estado", "cidade", "produto"], as_index=False)["preco_revenda"]
    .mean()
    .sort_values(["data_planilha", "estado", "cidade", "produto"])
)

resultado["preco_revenda"] = resultado["preco_revenda"].round(6)
resultado["data_planilha"] = resultado["data_planilha"].dt.strftime("%Y-%m-%d")
resultado.to_csv(output_path, index=False)

print(f"Arquivos processados: {len(arquivos)}")
print(f"Linhas consolidadas: {len(resultado)}")
print(f"CSV gerado em: {output_path}")

resultado.head(20)

Arquivos processados: 11
Linhas consolidadas: 16253
CSV gerado em: /home/mbmn/dev/ando/organizador/temp/combustiveis/saida/planilha_consolidada.csv


,data_planilha,estado,cidade,produto,preco_revenda
0,2026-01-03,ACRE,CRUZEIRO DO SUL,ETANOL,6.015000
1,2026-01-03,ACRE,CRUZEIRO DO SUL,GASOLINA ADITIVADA,8.090000
2,2026-01-03,ACRE,CRUZEIRO DO SUL,GASOLINA COMUM,7.975000
3,2026-01-03,ACRE,CRUZEIRO DO SUL,GLP,128.000000
4,2026-01-03,ACRE,RIO BRANCO,ETANOL,5.217000
5,2026-01-03,ACRE,RIO BRANCO,GASOLINA ADITIVADA,7.338333
6,2026-01-03,ACRE,RIO BRANCO,GASOLINA COMUM,7.274167
7,2026-01-03,ACRE,RIO BRANCO,GLP,122.782609
8,2026-01-03,ALAGOAS,ARAPIRACA,ETANOL,4.958182
9,2026-01-03,ALAGOAS,ARAPIRACA,GASOLINA ADITIVADA,6.420000
